# V3: LR/RF/XGBoost stacking

Compares static LR, RF, XGBoost and stacking on common outer folds.

The first level is three fixed-parameter Pipelines; there is no repeated grid tuning in the stacking sub-folds. The second-level LR is trained only on the first-level predicted probabilities on the stacking inner validation folds. Each base model redoes imputation, standardisation and LASSO selection on that inner training subset; features are not generated on the whole development set first, and training-set fitted probabilities are not passed off as out-of-fold predictions.

The outer validation patients never take part in the above. Each LR/RF/XGBoost experiment uses the same outer split; the second level has C=1 fixed in advance, passthrough=False; stacking inner 3 folds; LASSO inner 5 folds/30 λ. All base-model parameters are in config.yaml and were not re-chosen on the basis of the current 6 folds.

The new static baseline used for comparison shares the complete Pipeline with the extensions. It follows the V3 λ.1se rule, but also redoes preprocessing inside the LASSO folds and uses fixed base-model parameters, so it cannot be required to match the old V3 tables digit for digit.

**balanced simplified-compute version**: fixed base-model parameters, full keeps the 5×10 outer validation; LASSO inner 5 folds/30 λ, stacking inner 3 folds. Uses a cache separate from the parent strict version; the old predictions of the first 6 folds are not mixed in. No real patient model of the new version has been run yet.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the project from the extension directory or the notebooks subdirectory, to avoid hard-coding a Windows drive letter.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dn_extensions" / "settings.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("start from dn_mortality_repro_V3_extensions or its notebooks subdirectory.")
sys.path.insert(0, str(ROOT))
print("extension directory:", ROOT)


In [ ]:
# pilot is a small 1×3 outer-fold trial run; for the formal experiment set PROFILE to "full".
# Starts only when RUN_EXPERIMENT is set to True; full may take a long time.
PROFILE = "pilot"
RUN_EXPERIMENT = False
EXPERIMENTS = ['static_lr', 'static_rf', 'static_xgb', 'static_stacking']
if RUN_EXPERIMENT:
    from dn_extensions.evaluation import run_evaluation
    from dn_extensions.settings import profile_options
    output, summary, comparisons = run_evaluation(profile_options(PROFILE), EXPERIMENTS)
    display(summary)
    display(comparisons)
else:
    print("model training in this notebook has not been started.")


In [ ]:
# Show the formal full results first; without results print the status, no retraining.
RESULT_PROFILE = "full"
latest = ROOT / "results" / RESULT_PROFILE / "latest.txt"
if latest.exists():
    folder = latest.parent / latest.read_text().strip()
    summary = pd.read_csv(folder / "summary.csv")
    display(summary[summary.experiment.isin(EXPERIMENTS)])
    print("detailed record:", folder / "report.md")
else:
    print("no experiment results for this mode yet. Set RESULT_PROFILE to pilot to show pilot results.")


## Experiment status and conclusions (updated automatically)

Run mode: `full`; method: balanced fixed parameters; result directory: `results/full/27464ba98305edbc`.

LASSO 5 folds/30 λ, stacking 3 folds; static and extension models use the same simplified method. No folds of the strict version were mixed in.

The fixed held-out set took no part in this experiment. Tree-model extensions need no internal stacking cross-fitting; the original fixed parameters and within-training-fold preprocessing are unchanged.

The Holm correction covers the paired comparisons currently available; corrected p-values may change when experiments are added later.

```text

     experiment  n_outer_folds  AUROC_fold_mean  AUROC_fold_sd  AUPRC_fold_mean  Brier_fold_mean  calibration_slope_fold_mean  seconds_total  AUROC_pooled_mean  AUPRC_pooled_mean  Brier_pooled_mean  cal_intercept_pooled_mean  cal_slope_pooled_mean
      static_lr             50          0.79928        0.02648          0.60740          0.15092                      0.92073      502.87958            0.79885            0.59482            0.15091                    0.00385                0.89831
      static_rf             50          0.79798        0.02809          0.60618          0.15177                      1.19841       15.22184            0.79671            0.59435            0.15177                    0.00179                1.16211
static_stacking             50          0.80466        0.02657          0.61586          0.15048                      1.07477     1319.71249            0.80325            0.60386            0.15047                    0.00134                1.05015
     static_xgb             50          0.79504        0.02965          0.60572          0.15250                      0.94083        4.61352            0.79408            0.59662            0.15249                    0.00766                0.91690

```

Paired differences:

```text

     experiment reference metric  mean_diff  sd_diff  se_corrected       lo      hi        t  df  p_value  k  r  p_holm_current_family
      static_rf static_lr  AUROC   -0.00130  0.01457       0.00528 -0.01190 0.00931 -0.24552  49  0.80708 10  5                    1.0
      static_rf static_lr  AUPRC   -0.00122  0.02900       0.01050 -0.02232 0.01989 -0.11569  49  0.90837 10  5                    1.0
      static_rf static_lr  Brier    0.00085  0.00469       0.00170 -0.00256 0.00427  0.50207  49  0.61787 10  5                    1.0
     static_xgb static_lr  AUROC   -0.00423  0.01409       0.00510 -0.01449 0.00602 -0.82968  49  0.41074 10  5                    1.0
     static_xgb static_lr  AUPRC   -0.00167  0.02894       0.01048 -0.02273 0.01938 -0.15972  49  0.87376 10  5                    1.0
     static_xgb static_lr  Brier    0.00158  0.00519       0.00188 -0.00220 0.00536  0.83926  49  0.40540 10  5                    1.0
static_stacking static_lr  AUROC    0.00539  0.00823       0.00298 -0.00060 0.01138  1.80933  49  0.07654 10  5                    1.0
static_stacking static_lr  AUPRC    0.00847  0.01519       0.00550 -0.00259 0.01952  1.53947  49  0.13012 10  5                    1.0
static_stacking static_lr  Brier   -0.00044  0.00289       0.00105 -0.00254 0.00166 -0.41929  49  0.67683 10  5                    1.0

```

- `static_rf` vs `static_lr`: ΔAUROC=-0.0013, corrected 95% CI [-0.0119, 0.0093]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `static_xgb` vs `static_lr`: ΔAUROC=-0.0042, corrected 95% CI [-0.0145, 0.0060]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `static_stacking` vs `static_lr`: ΔAUROC=0.0054, corrected 95% CI [-0.0006, 0.0114]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

## 2026-10-14 review and interpretation of results

This run did complete 50 outer folds for each of the four model groups, seeds 42–46. Only the V3 development set of 2045 patients and 544 deaths was used, and the fixed held-out labels were not read. The fixed-parameter balanced method is recorded separately from the parent strict version.

| Model | AUROC fold mean | AUPRC pooled OOF mean | Brier pooled OOF mean | Calibration slope pooled OOF mean |
|---|---:|---:|---:|---:|
| static_lr | 0.7993 | 0.5948 | 0.1509 | 0.898 |
| static_rf | 0.7980 | 0.5944 | 0.1518 | 1.162 |
| static_xgb | 0.7950 | 0.5966 | 0.1525 | 0.917 |
| static_stacking | 0.8047 | 0.6039 | 0.1505 | 1.050 |

**Main judgement: stacking shows a small, fairly consistent improvement, but the evidence is not enough to claim a clear performance gain.**

- AUROC fold mean goes from 0.7993 to 0.8047. Paired ΔAUROC=0.0054, Nadeau–Bengio corrected 95% CI [-0.0006, 0.0114], p=0.0765; Holm-corrected p over the current three AUROC comparisons = 0.2296. The interval crosses 0, so "significantly improved" cannot be written.
- For the pooled OOF AUROC of each repeat, stacking is higher than LR in all 5; across the 50 outer folds it is higher than LR in 41 and lower in 9. This shows a consistent direction, but these results share patients, so the 50 folds cannot be treated as 50 independent experiments for an ordinary significance test.
- The mean pooled OOF AUROC is LR 0.7989 and stacking 0.8032; this is a different summary from the fold mean, and the report must say which is used.
- AUPRC also improves slightly, but the fold-paired Δ=0.0085, corrected 95% CI [-0.0026, 0.0195], p=0.1301, still does not support a clear gain.
- The point estimate of the stacking calibration slope is closer to 1 (about 1.05, versus about 0.90 for LR), and Brier is only slightly lower. This is a descriptive result and does not prove that the calibration improvement is statistically significant.
- DCA also does not show stacking consistently better than LR across the threshold range; of 56 thresholds, the mean net benefit is higher at 31 and lower at 25. A higher AUROC alone therefore cannot support a claim of overall improvement in clinical utility.

**Run time:** the recorded fit/predict times of all models add up to about 30.7 minutes. Stacking averages 26.4 seconds per fold. This was measured while running alongside the strict version; RF and XGBoost reused the common representation cache, so their separate timings do not include the representation-learning cost already paid by LR.

**Next step:** keep this balanced setting unchanged and run the temporal-trajectory and burden/embedding ablations separately, comparing on the same outer folds and the same static baseline. Do not change hyperparameters ad hoc based on results, or repeatedly look at the fixed held-out set, because this round is not yet significant.

The 0.8047 above is a repeated outer-CV mean AUROC, not a single held-out result, and cannot be used directly as a confirmatory advantage over the original paper's 0.780.


## Multiple-comparison update (2026-10-14)

After adding two ablations, each metric now has 6 paired comparisons. The ΔAUROC, confidence interval and Holm-uncorrected p-value of stacking are unchanged; the Holm-corrected p is updated to 0.4592. The values in the earlier review record correspond to the 3 comparisons of that time. For the formal summary use the latest paired_comparisons.csv.

# Final development-set ablation record for stacking (2026-10-14)

The planned 5 repeats × 10 folds, 50 folds in total, seeds 42–46, are complete. Development set of 2,045 patients, 544 deaths. Paired comparisons on the same validation folds; the fixed held-out labels were not read and no additional training was done. This complete record replaces the provisional conclusion from the first four repeats; the interim snapshot is kept for traceability.

**Conclusion: static stacking shows a small numerical gain over the current best single model, LR, in a consistent direction across the five repeats, but it is not statistically significant. Adding temporal trajectories and prior-diagnosis representations together shows no additional benefit.**

| Model | AUROC (50-fold mean) | AUPRC (50-fold mean) | Brier (50-fold mean, lower is better) |
|---|---:|---:|---:|
| Static LR | 0.799275 | 0.607397 | 0.150918 |
| Static RF | 0.797980 | 0.606182 | 0.151771 |
| Static XGBoost | 0.795041 | 0.605724 | 0.152496 |
| Static stacking | 0.804665 | 0.615863 | 0.150479 |
| Combined-extension stacking | 0.804750 | 0.615715 | 0.150381 |

These are computed per outer validation fold first and then averaged over the 50 folds. The other summary, "per repeat, stitch the OOF probabilities back into the complete development set, compute the metric, then average over the five repeats", gives AUROCs of: static LR 0.798852, static stacking 0.803249, combined stacking 0.803473. Do not mix the two summaries.

## Static stacking versus single models

Relative to static LR, the 50-fold mean AUROC increases by 0.005389 (about 0.54 percentage points), AUPRC increases by 0.008466 and Brier decreases by 0.000439. The mean AUROC and AUPRC of static stacking are also higher than those of the current fixed-parameter RF and XGBoost; this is not taken to mean that it beats every tuned single model.

The pooled OOF AUROC and AUPRC are higher than LR in all five repeats; AUROC is higher in 41 of the 50 folds. The same development set is reused, so these are not five independent external validation samples.

The Nadeau–Bengio corrected 95% interval of the paired AUROC difference is [−0.000596, +0.011375], p=0.076535; after Holm correction over the current 7 AUROC comparisons p=0.535748. AUPRC difference interval [−0.002585, +0.019518], p=0.130124; Brier difference interval [−0.002541, +0.001663], p=0.676835. It can therefore be described as "a small numerical gain, consistent in direction across repeated splits", but not as "significantly better than LR". The difference interval here is not a confidence interval for the AUROC of the stacking model itself.

## Does the combined extension improve stacking further?

Relative to static stacking, the combined model increases the 50-fold mean AUROC by only 0.000086 (about 0.009 percentage points), AUPRC decreases by 0.000148 and Brier decreases by 0.000099; the values are practically very close.

The 95% interval of the paired AUROC difference is [−0.007569, +0.007740], p=0.982179, Holm p=1. The pooled AUROC rises in three of the five repeats and falls in two; it rises in 30 of the 50 folds. The number of folds won cannot prove a gain, and non-significance cannot prove strict equivalence.

| seed | Static stacking − LR AUROC | Combined − static stacking AUROC |
|---|---:|---:|
| 42 | +0.003559 | -0.001830 |
| 43 | +0.005730 | -0.000344 |
| 44 | +0.004561 | +0.000316 |
| 45 | +0.004749 | +0.001964 |
| 46 | +0.003383 | +0.001015 |

## Limits for reporting and further experiments

For now static stacking can be treated as a candidate model, with LR kept as a simple and competitive comparator. For the combined model, report that "the current extensions bring no clear additional gain"; there is no need to claim an improvement for a mean difference close to zero. The numerical gain may come from complementary base-model predictions, but this metric comparison does not directly verify the mechanism.

This round is the balanced version with fixed base-model parameters and simplified inner computation, and its results are exploratory development-set results. They cannot be compared directly with the paper's 0.780 to claim that the paper is surpassed, because the cohort and evaluation differ. If the fixed held-out set has already been looked at, it cannot be used as a brand-new untouched test set for choosing extensions either.

The separate temporal, diagnosis-burden and embedding ablations have so far been evaluated only for LR. In combined stacking the LR, RF and XGBoost all received the combined extensions, but the new-feature results of RF and XGBoost have not been reported separately. "No benefit from combining" therefore cannot be generalised to "temporal features or embeddings are useless for every model". The four diagnosis-burden indicators and the ICD embedding are appended directly after the numeric LASSO screening and are not excluded by LASSO; the temporal features take part in the numeric screening.

What can be said at the meeting:

> Over the complete five repeats of ten folds, the mean AUROC of static stacking rises from 0.799 for LR to 0.805 and is higher in all five repeats, but the interval of the paired difference still includes zero, so the gain is not yet statistically confirmed. With temporal trajectories and diagnosis representations added it is still about 0.805, with no additional gain. For now static stacking is a promising candidate and LR remains an important comparator.
